# Used Cars Price Prediction with XGBRegressor 

## Project Description
This project focuses on building a predictive model for estimating used car prices using the XGBoost regression algorithm. The approach incorporates data preprocessing, feature engineering, model training with hyperparameter optimization, and model evaluation. The project utilizes various techniques, including label encoding, one-hot encoding, and feature scaling, to process the dataset effectively.

#### Data Preparation and Preprocessing:
* **Loading the Data**: The dataset (`used_cars_clean.csv`) is loaded using pandas.
* **Initial Exploration**: Basic data exploration using `info()`, `nunique()`, and `head()` methods to understand the structure and unique values.
* **Column Cleaning**: The dataset is cleaned by dropping unnecessary columns, such as an unnamed column.
* **Feature Encoding and Scaling**:
    - **Label Encoding**: Applied to categorical columns like `make`, `body-style`, `drive-wheels`, `engine-type`, `num-of-cylinders`, and `fuel-system`.
    - **One-Hot Encoding**: Applied to columns such as `num-of-doors`, `engine-location`, and `horsepower-binned`.
    - **Feature Scaling**: Standardized or MinMax-scaled numeric columns (e.g., `normalized-losses`, `wheel-base`, `length`, etc.), excluding the target column (`price`).

#### Model Training (XGBoost with Hyperparameter Optimization):
* **Model Selection**: The XGBoost Regressor (`XGBRegressor`) is used for building the predictive model.
* **Hyperparameter Tuning**:
    - **RandomizedSearchCV**: Used for tuning hyperparameters such as `max_depth`, `n_estimators`, `learning_rate`, and `reg_lambda`.
    - **Training Process**: Model training is performed using the training set, and the best hyperparameters are identified based on R? score using cross-validation.
* **Model Training Results**: The best model is selected and evaluated based on training and test R? scores.

#### Evaluation:
* **Metrics Calculation**: R? scores are calculated for training, validation, and testing datasets to assess model performance.
* **Feature Importance**: The importance of each feature is visualized using the `plot_importance()` function to understand which features have the greatest impact on predictions.
* **Residual Analysis**: A histogram of residuals is plotted to inspect the distribution of prediction errors.
* **Actual vs Predicted Visualization**: A scatter plot of actual vs. predicted values is created to visually evaluate prediction accuracy.


In [ ]:
from xgboost import XGBRegressor, plot_importance
from sklearn.metrics import make_scorer, r2_score
from sklearn.model_selection import RandomizedSearchCV, validation_curve, learning_curve, train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
import seaborn as sns; sns.set()
# Suppress the SettingWithCopyWarning
pd.set_option('mode.chained_assignment', None)
plt.rcParams['figure.figsize'] = (6, 4)

In [ ]:
# Load the data
df = pd.read_csv('used_cars_clean.csv')

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df.nunique()

## Preprocessing

In [ ]:
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
# Columns to encode with Label Encoding
label_encode_columns = [
    'make', 
    'body-style', 
    'drive-wheels', 
    'engine-type', 
    'num-of-cylinders', 
    'fuel-system'
]

one_hot_encoding_columns = [
    'num-of-doors',
    'engine-location',
    'horsepower-binned',
]

# Columns to scale
scale_columns = ['normalized-losses', 'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-size', 'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg']

# 1. Label Encoding for specified columns
le = LabelEncoder()
for col in label_encode_columns:
    if col != 'price':  
        df[col] = le.fit_transform(df[col])

# 2. One-Hot Encoding for specified columns
df = pd.get_dummies(df, columns=one_hot_encoding_columns, drop_first=True)

# 3. Scaling the numeric columns (excluding 'price')
# scaler = StandardScaler()  # Or use MinMaxScaler() if preferred
# df[scale_columns] = scaler.fit_transform(df[scale_columns])

In [ ]:
df

In [ ]:
df.info()

## Split the data into features and target

In [ ]:
X = df.drop(columns=['price'])

In [ ]:
y = df['price']

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4700)

## XGBRegressor Model

In [ ]:
xgb = XGBRegressor()

In [ ]:
param_distributions = {
    'max_depth': range(1, 7),
    'n_estimators': range(10, 201, 10),
    'learning_rate': np.linspace(0.001, 0.05, 35),
    'reg_lambda': np.linspace(0.05, 2.0, 50),
}

In [ ]:
random_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_distributions,
    n_iter=200,
    scoring=make_scorer(r2_score),
    cv=5,
    n_jobs=-1,
    random_state=4700
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
print('Best hyperparameters:', random_search.best_params_)
print('Best R^2:', random_search.best_score_)

### Feature Importance

In [ ]:
plot_importance(best_model)
plt.show()

### Training and Testing Scores

In [ ]:
#predict
y_train_pred = best_model.predict(X_train)
y_pred = best_model.predict(X_test)
print('Training R^2:', r2_score(y_train, y_train_pred))
print('Validation R^2:', random_search.best_score_)
print('Testing R^2:', r2_score(y_test, y_pred))

## Plotting Testing Results

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.8)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted Values")
plt.show()

In [ ]:
residuals = y_test - y_pred
plt.hist(residuals, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Histogram of Residuals")
plt.show()